# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset contains mass spectrometry protein abundance data derived from extracellular vesicles and is described by a Croissant schema, accessible via URL. All entities are referenced by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, data fields, and their `@id`s.

We'll print all record sets in the dataset and, for one record set, show the available field `@id`s.

In [ ]:
# List all available record sets (referenced by their @id)
print("Available record sets (by @id):")
record_set_ids = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]
for rset in record_set_ids:
    print(f"- {rset}")

if len(record_set_ids) == 0:
    print("No recordSet listed in metadata. Using dataset.records() to enumerate available record sets:")
    # mlcroissant will fallback and infer record set IDs; but the schema should enumerate them
    try:
        # record_set ids inferred from dataset.records iterator
        records_iter = getattr(dataset, "_metadata")._records()
        inferred_sets = set()
        for rec in records_iter:
            inferred_sets.add(rec["@type"])
        for rset in inferred_sets:
            print(f"- {rset}")
    except Exception as e:
        print("Could not infer record sets.")

In [ ]:
# List direct example records and print the fields present for a record set by its @id
# You must substitute this for a known record set @id in the dataset.
# As the metadata does not enumerate recordSets, we'll iterate all available and pick the first.
if record_set_ids:
    example_record_set = record_set_ids[0]
else:
    # fallback to commonly used 'ProteinAbundance' if known (otherwise you will need to consult the schema)
    # Replace with actual record set @id as needed
    example_record_set = None

if example_record_set is not None:
    print(f"\nFields in record set {example_record_set}:")
    for record in dataset.records(record_set=example_record_set):
        print(record)
        break  # Print only first record as an example
else:
    # Try default record set if schema is not explicit
    try:
        for record in dataset.records():
            print(record)
            break
    except Exception as e:
        print("No records available. Check the schema record sets.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# In this dataset, record sets are likely to be provided as one main table.
# Let's extract all records (assuming only one main record set)
record_sets = record_set_ids if len(record_set_ids) else []
dataframes = {}

if not record_sets:
    # Try to load without specifying record_set, fallback for single-table datasets
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records.")
        print(df.columns.tolist())
        display(df.head())
        # Save for later EDA/visualization
        dataframes[None] = df
    except Exception as e:
        print("Could not extract any records.", e)
else:
    for record_set in record_sets:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from {record_set}.")
        print(f"Fields (columns) in {record_set}:", df.columns.tolist())
        display(df.head())
        dataframes[record_set] = df


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping data. Use actual field `@id`s or column names identified in the previous section.

In [ ]:
# Pick a record set (or None if only one)
selected_record_set = record_sets[0] if record_sets else None
df = dataframes[selected_record_set]

# Show numeric fields for EDA
numeric_fields = df.select_dtypes(include='number').columns.tolist()
if numeric_fields:
    print("Numeric fields available:", numeric_fields)
    # Use the first as example
    numeric_field = numeric_fields[0]
else:
    print("No numeric fields detected.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a categorical field if available
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}, mean of {numeric_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group/categorical field for grouping found.")
else:
    print("Unable to perform EDA as no numeric field is present.")

## 5. Visualization
Visualize data distributions or relationships. Below, we will plot the distribution of the numeric field if available, and a scatterplot between two numeric fields (if two exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_fields[0]].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_fields[0]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel("Frequency")
    plt.show()

    # If there's a second numeric field, do a scatterplot
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6,5))
        sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs. {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()

else:
    print("No numeric fields to plot.")

## 6. Conclusion
This notebook showed how to use the `mlcroissant` library to explore the FAIR^2 dataset containing mass spectrometry-derived protein abundance measurements from human extracellular vesicles. After loading the dataset via its Croissant schema and examining available fields and record sets, we've performed basic exploratory data analysis and simple visualizations. Further analysis can be performed using domain-specific questions and field knowledge.

_All dataset entities referenced in this notebook were handled by their `@id`s when available. For more details, always consult the original Croissant schema._